# Evaluate ai.translate(...) Quality with PySpark

This notebook evaluates translation for structure, content preservation, and natural target-language output. The AI transformations and LLM-as-a-Judge evaluation remain in Spark. Only the small, materialized result set is converted to pandas for metrics and charts.

### What You'll Do
1. Translate multilingual sample text to English.
2. Score coherence, consistency, and translation quality.
3. Inspect average and per-sample quality.
4. Compare the baseline with an explainable structured translation prompt.

### Before You Start
- **Runtime** - This notebook was made for **Fabric 1.3 runtime**.
- **Customize it** - Replace the sample data and adapt the judge criteria to your use case.
- **Keep comparisons fair** - Hold the judge model and prompts fixed while changing one executor setting at a time.
- **Validate important decisions** - LLM judge scores are useful proxies, not a substitute for human-reviewed production samples.

| Metric | Measures |
|--------|----------|
| **Coherence** | Structure and flow are appropriate for the target language |
| **Consistency** | Source content is preserved without additions or omissions |
| **Translation quality** | Meaning is accurate and the result reads naturally |

[ai.translate PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/translate)


## 1. Setup

Install Pydantic for the structured response schemas used by the judge.
The baseline uses `gpt-5-mini` with low reasoning effort. A fixed
`gpt-5.1` judge evaluates every translation variant.
See the [AI Functions model and CU rate table](https://aka.ms/aifunctions-fabric-llm-cu-rates)
for all available models.


In [ ]:
%pip install -q pydantic 2>/dev/null


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import synapse.ml.spark.aifunc as aifunc
from pydantic import BaseModel, Field
from pyspark.sql import functions as F

# Use the smaller model for the function under test and a larger fixed model for judging.
EXECUTOR_OPTIONS = {
    "deploymentName": "gpt-5-mini",
    "reasoningEffort": "low",
}
JUDGE_OPTIONS = {
    "deploymentName": "gpt-5.1",
    "reasoningEffort": "medium",
}

class MetricEval(BaseModel):
    reason: str = Field(description="Brief rationale for the score")
    score: int = Field(ge=1, le=5, description="Integer score from 1 to 5")

def materialize(frame):
    cached = frame.cache()
    _ = cached.count()
    error_columns = [name for name in cached.columns if name.endswith("_error")]
    if error_columns:
        has_error = F.lit(False)
        for name in error_columns:
            has_error = has_error | F.coalesce(
                F.length(F.trim(F.col(name).cast("string"))) > 0,
                F.lit(False),
            )
        failed_rows = cached.filter(has_error)
        failure_count = failed_rows.count()
        if failure_count:
            print(f"{failure_count} row(s) contain AI Function errors:")
            id_columns = [
                name for name in ("sample_id", "ticket_id") if name in cached.columns
            ]
            display(failed_rows.select(*id_columns, *error_columns))
    return cached

def fresh_ai_view(frame):
    return frame.select("*")

def add_judge_metric(frame, metric_name, prompt, column_prefix=""):
    score_col = f"{column_prefix}{metric_name}"
    response_col = f"_{score_col}_response"
    raw_score_col = f"_{score_col}_raw_score"
    error_col = f"_{score_col}_error"
    judged = fresh_ai_view(frame).ai.generate_response(
        prompt=prompt,
        is_prompt_template=True,
        output_col=response_col,
        error_col=error_col,
        response_format=MetricEval,
        **JUDGE_OPTIONS,
    )
    invalid_score = (
        F.col(raw_score_col).isNull()
        | ~F.col(raw_score_col).between(1, 5)
        | (F.col(raw_score_col) != F.floor(F.col(raw_score_col)))
    )
    validation_message = "Judge score must be an integer from 1 to 5"
    return (
        judged
        .withColumn(
            raw_score_col,
            F.get_json_object(F.col(response_col), "$.score").cast("double"),
        )
        .withColumn(
            error_col,
            F.when(
                invalid_score,
                F.when(
                    F.length(
                        F.trim(F.coalesce(F.col(error_col), F.lit("")))
                    ) > 0,
                    F.concat(
                        F.col(error_col),
                        F.lit(f"; {validation_message}"),
                    ),
                ).otherwise(F.lit(validation_message)),
            ).otherwise(F.col(error_col)),
        )
        .withColumn(
            score_col,
            F.when(
                ~invalid_score,
                F.col(raw_score_col).cast("int"),
            ),
        )
        .withColumn(
            f"{score_col}_reason",
            F.get_json_object(F.col(response_col), "$.reason"),
        )
        .drop(raw_score_col)
    )


## 2. Load Sample Data


In [ ]:
TARGET_LANGUAGE = "English"
rows = [(1,
  'Tutti gli articoli acquistati dal nostro negozio online possono essere restituiti entro 30 giorni di '
  'calendario dalla data di consegna originale. Per avere diritto a un rimborso completo, la merce deve '
  'essere nella confezione originale con tutte le etichette attaccate e non deve mostrare segni di usura o '
  'danni. I rimborsi vengono elaborati sul metodo di pagamento originale entro 5-7 giorni lavorativi dopo la '
  "ricezione e l'ispezione dell'articolo restituito."),
 (2,
  'Nous vous rappelons que votre rendez-vous avec le Dr. Elena Vasquez est prévu pour jeudi 14 mars à 14h30 '
  'dans la suite 410 du Westfield Medical Plaza. Veuillez arriver 15 minutes en avance pour remplir vos '
  "documents d'admission. Apportez votre carte d'assurance, une pièce d'identité avec photo et la liste de "
  'tous les médicaments que vous prenez actuellement, y compris le dosage et la fréquence.'),
 (3,
  'Durch den Zugang zu oder die Nutzung dieses Dienstes erklärt sich der Nutzer mit diesen Allgemeinen '
  'Geschäftsbedingungen einverstanden. Das Unternehmen behält sich das Recht vor, diese Bedingungen '
  'jederzeit ohne vorherige Ankündigung zu ändern; die fortgesetzte Nutzung des Dienstes nach solchen '
  'Änderungen gilt als Zustimmung des Nutzers zu den geänderten Bedingungen. Streitigkeiten aus dieser '
  'Vereinbarung unterliegen den Gesetzen des Staates Delaware und werden ausschließlich vor den Gerichten '
  'von Wilmington verhandelt.'),
 (4,
  'Welcome to Acme Analytics! Your team account has been successfully created. To get started, invite your '
  'colleagues from the Settings > Team Members page - each member will receive an activation email valid for '
  '48 hours. We recommend connecting your first data source within the onboarding wizard, which supports CSV '
  'uploads, direct database connections via JDBC, and REST API integrations out of the box.'),
 (5,
  'Nossa API pública aplica um limite de 1.000 requisições por minuto por chave de API. Se você exceder esse '
  'limite, o servidor responderá com HTTP 429 (Too Many Requests) e incluirá um cabeçalho Retry-After '
  'indicando o número de segundos a aguardar antes de tentar novamente. Para necessidades de maior '
  'throughput, considere fazer upgrade para nosso plano Enterprise, que oferece limites configuráveis de até '
  '50.000 requisições por minuto e isolamento dedicado de endpoint.')]
df = spark.createDataFrame(rows, ["sample_id", "text"])
df = df.withColumn("_target_lang", F.lit(TARGET_LANGUAGE))
display(df.select("sample_id", "text", "_target_lang"))


## 3. Run `ai.translate`


In [ ]:
translation_df = materialize(
    df.ai.translate(
        to_lang=TARGET_LANGUAGE.lower(),
        input_col="text",
        output_col="translation",
        error_col="executor_error",
        **EXECUTOR_OPTIONS,
    )
)
display(translation_df.select("text", "translation"))
display(translation_df.ai.stats)


## 4. Evaluate with an LLM Judge


In [ ]:
EVAL_METRICS = {
    "coherence": """Score translation coherence from 1 to 5.
A score of 5 means the translated structure and flow are natural while preserving
the source organization where appropriate.

<target_language>
{_target_lang}
</target_language>
<source>
{text}
</source>
<translation>
{translation}
</translation>""",
    "consistency": """Score content preservation from 1 to 5.
A score of 5 means all source information is translated without additions,
omissions, or changed facts.

<target_language>
{_target_lang}
</target_language>
<source>
{text}
</source>
<translation>
{translation}
</translation>""",
    "translation_quality": """Score accuracy and naturalness from 1 to 5.
A score of 5 means the translation conveys the correct meaning and reads naturally
in the target language, including domain terminology.

<target_language>
{_target_lang}
</target_language>
<source>
{text}
</source>
<translation>
{translation}
</translation>""",
}

evaluated_df = fresh_ai_view(translation_df)
for metric_name, prompt in EVAL_METRICS.items():
    evaluated_df = add_judge_metric(evaluated_df, metric_name, prompt)
evaluated_df = materialize(evaluated_df)
display(evaluated_df.select("text", "translation", *EVAL_METRICS.keys()))


## 5. Results


In [ ]:
METRICS = ['coherence', 'consistency', 'translation_quality']
results_pd = evaluated_df.select('sample_id', 'text', 'translation', 'coherence', 'consistency', 'translation_quality').toPandas()

score_summary = pd.DataFrame({
    "Metric": ['Coherence', 'Consistency', 'Translation Quality'],
    "Average score": [results_pd[metric].mean() for metric in METRICS],
    "Scored rows": [results_pd[metric].notna().sum() for metric in METRICS],
})
def quality_status(score, is_complete):
    if not is_complete or pd.isna(score):
        return "INCOMPLETE"
    return "PASS" if score >= 4 else "REVIEW" if score >= 3.5 else "FAIL"

score_summary["Status"] = [
    quality_status(score, scored_rows == len(results_pd))
    for score, scored_rows in zip(
        score_summary["Average score"],
        score_summary["Scored rows"],
    )
]
display(score_summary.round(2))

labels = score_summary["Metric"].tolist()
values = score_summary["Average score"].tolist()
fig = plt.figure(figsize=(13, 4.5))
bar_ax = fig.add_subplot(1, 2, 1)
bars = bar_ax.bar(labels, values, color="#0077aa")
bar_ax.set_ylim(0, 5)
bar_ax.set_ylabel("Score (1-5)")
bar_ax.set_title('Translation Quality')
bar_ax.axhline(y=4, color="#999999", linestyle="--", alpha=0.6)
bar_ax.tick_params(axis="x", rotation=20)
bar_ax.bar_label(bars, fmt="%.2f", padding=2)

if len(METRICS) >= 3:
    detail_ax = fig.add_subplot(1, 2, 2, polar=True)
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    detail_ax.plot(
        angles + angles[:1],
        values + values[:1],
        "o-",
        linewidth=2,
        color="#9955bb",
    )
    detail_ax.fill(
        angles + angles[:1],
        values + values[:1],
        alpha=0.25,
        color="#9955bb",
    )
    detail_ax.set_xticks(angles)
    detail_ax.set_xticklabels(labels)
    detail_ax.set_ylim(0, 5)
    detail_ax.set_title("Quality Profile", pad=20)
else:
    detail_ax = fig.add_subplot(1, 2, 2)
    detail_ax.hist(
        [results_pd[metric].dropna() for metric in METRICS],
        bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
        label=labels,
        alpha=0.7,
    )
    detail_ax.set_xticks([1, 2, 3, 4, 5])
    detail_ax.set_xlabel("Score")
    detail_ax.set_ylabel("Rows")
    detail_ax.set_title("Score Distribution")
    detail_ax.legend()
plt.tight_layout()
plt.show()

results_pd["scored_metrics"] = results_pd[METRICS].notna().sum(axis=1)
complete_rows = results_pd["scored_metrics"].eq(len(METRICS))
results_pd["average_score"] = (
    results_pd[METRICS].mean(axis=1).where(complete_rows).round(2)
)
results_pd["status"] = [
    quality_status(score, is_complete)
    for score, is_complete in zip(
        results_pd["average_score"],
        complete_rows,
    )
]


In [ ]:
breakdown_pd = results_pd[
    [
        "text",
        "translation",
        "coherence",
        "consistency",
        "translation_quality",
        "scored_metrics",
        "average_score",
        "status",
    ]
].copy()
breakdown_pd["text"] = breakdown_pd["text"].str[:100] + "..."
breakdown_pd["translation"] = breakdown_pd["translation"].str[:100] + "..."
display(breakdown_pd)


## 6. Optional Refinement: Explainable Translation

Compare `ai.translate` with a structured custom prompt that returns both the
translation and a short explanation. Both variants use the same judge criteria.


In [ ]:
class ExplainableTranslation(BaseModel):
    reason: str = Field(description="Brief explanation of important translation choices")
    translation: str = Field(description="Translation in the requested target language")

CUSTOM_PROMPT = """Translate the source text into {_target_lang}.
Preserve all facts, technical terms, numbers, and paragraph structure where appropriate.
Return the translation and a brief explanation of important choices.

<source>
{text}
</source>"""

custom_df = fresh_ai_view(translation_df).ai.generate_response(
    prompt=CUSTOM_PROMPT,
    is_prompt_template=True,
    output_col="_custom_response",
    error_col="_custom_error",
    response_format=ExplainableTranslation,
    **EXECUTOR_OPTIONS,
)
custom_df = (
    custom_df
    .withColumn(
        "custom_translation",
        F.get_json_object(F.col("_custom_response"), "$.translation"),
    )
    .withColumn(
        "custom_reason",
        F.get_json_object(F.col("_custom_response"), "$.reason"),
    )
)
custom_df = materialize(custom_df)

custom_eval_df = fresh_ai_view(custom_df)
for metric_name, prompt in EVAL_METRICS.items():
    custom_eval_df = add_judge_metric(
        custom_eval_df,
        metric_name,
        prompt.replace("{translation}", "{custom_translation}"),
        column_prefix="custom_",
    )
custom_eval_df = materialize(custom_eval_df)
display(
    custom_eval_df.select(
        "text", "translation", "custom_translation", "custom_reason"
    )
)


In [ ]:
custom_pd = custom_eval_df.select(
    "sample_id",
    "custom_coherence",
    "custom_consistency",
    "custom_translation_quality",
).toPandas()
comparison_rows_pd = (
    results_pd[["sample_id", "coherence", "consistency", "translation_quality"]]
    .merge(
        custom_pd,
        on="sample_id",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)
required_columns = [
    "coherence",
    "consistency",
    "translation_quality",
    "custom_coherence",
    "custom_consistency",
    "custom_translation_quality",
]
paired_mask = (
    comparison_rows_pd["_merge"].eq("both")
    & comparison_rows_pd[required_columns].notna().all(axis=1)
)
paired_count = int(paired_mask.sum())
excluded_count = int((~paired_mask).sum())
print(f"Paired rows: {paired_count} | Excluded rows: {excluded_count}")
if excluded_count:
    display(
        comparison_rows_pd.loc[
            ~paired_mask,
            ["sample_id", "_merge", *required_columns],
        ]
    )
if not paired_count:
    raise ValueError("No rows have complete baseline and custom translation scores.")
paired_pd = comparison_rows_pd.loc[paired_mask]

comparison_pd = pd.DataFrame({
    "Metric": ["Coherence", "Consistency", "Translation quality", "Overall"],
    "Baseline": [
        paired_pd["coherence"].mean(),
        paired_pd["consistency"].mean(),
        paired_pd["translation_quality"].mean(),
        paired_pd[
            ["coherence", "consistency", "translation_quality"]
        ].mean(axis=1).mean(),
    ],
    "Custom": [
        paired_pd["custom_coherence"].mean(),
        paired_pd["custom_consistency"].mean(),
        paired_pd["custom_translation_quality"].mean(),
        paired_pd[
            ["custom_coherence", "custom_consistency", "custom_translation_quality"]
        ].mean(axis=1).mean(),
    ],
})
comparison_pd["Delta"] = comparison_pd["Custom"] - comparison_pd["Baseline"]
display(comparison_pd.round(2))

plot_pd = comparison_pd[comparison_pd["Metric"] != "Overall"].set_index("Metric")
ax = plot_pd[["Baseline", "Custom"]].plot.bar(
    figsize=(8, 4),
    color=["#0077aa", "#22cc77"],
    rot=0,
)
ax.set_ylim(0, 5)
ax.set_ylabel("Average score (1-5)")
ax.set_title("Baseline vs Custom Translation")
ax.axhline(y=4, color="#999999", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


## Interpreting Results

| Average score | Suggested action |
|---------------|------------------|
| **4.5-5.0** | Strong candidate for production validation |
| **4.0-4.4** | Good; inspect the lowest-scoring samples |
| **3.5-3.9** | Acceptable for iteration; refine data, prompts, or labels |
| **Below 3.5** | Investigate before broader use |
| **INCOMPLETE** | One or more judge scores are missing; inspect AI Function errors |

| Metric or issue | Likely cause | Next step |
|-----------------|--------------|-----------|
| Coherence | Literal structure sounds unnatural | Allow target-language restructuring |
| Consistency | Content is missing or added | Tighten preservation requirements |
| Translation quality | Terminology or phrasing is weak | Add domain context and native review |

### Improving Quality

- Add domain terminology and target-register requirements to a custom prompt.
- Review consistency failures for omissions, added content, or changed numbers.
- Evaluate each target language with representative native-speaker samples.

Keep the judge configuration fixed for comparisons, and confirm release decisions with representative human-reviewed samples.

## Learn More

- [ai.translate PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/translate)
- [AI Functions overview](https://learn.microsoft.com/fabric/data-science/ai-functions/overview)
